# Preterm Birth Prediction – Classic ML Model

XGBoost pipeline on hand-crafted EHG features.

**Steps:**
1. Load CSV files
2. Merge & feature engineering (uterine coordination ratios)
3. Numeric cleanup & X/y split
4. Feature selection (SelectKBest, k=15)
5. Train/test split & StandardScaler
6. SMOTE-Tomek class balancing
7. XGBoost training
8. Threshold optimisation & evaluation

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, fbeta_score, precision_recall_curve
from imblearn.combine import SMOTETomek
from sklearn.impute import KNNImputer
from sklearn.feature_selection import RFE
import os

In [ ]:
# 1. טעינת הקבצים
try:
    from google.colab import drive
    drive.mount('/content/drive')
    path_features = '/content/drive/MyDrive/deep_birth/data/classic_ml_features.csv'
    path_meta = '/content/drive/MyDrive/deep_birth/data/tpehg_metadata.csv'
    path_clinical = '/content/drive/MyDrive/deep_birth/data/tpehg_clinical_data.csv'
except ImportError:
    path_features = r'C:\Users\yuval\Desktop\deep-birth\data\classic_ml_features.csv'
    path_meta = r'C:\Users\yuval\Desktop\deep-birth\data\tpehg_metadata.csv'
    path_clinical = r'C:\Users\yuval\Desktop\deep-birth\data\tpehg_clinical_data.csv'

df_features = pd.read_csv(path_features)
df_meta = pd.read_csv(path_meta)
df_clinical = pd.read_csv(path_clinical)

# 2. מיזוג נתונים
cols_to_drop = ['Gestation', 'Rectime', 'Age']
df_clinical_clean = df_clinical.drop(columns=[c for c in cols_to_drop if c in df_clinical.columns])
df_merged_meta = pd.merge(df_meta, df_clinical_clean, on='RecID', how='left')
df = pd.concat([df_features.drop('Label', axis=1, errors='ignore'), df_merged_meta], axis=1)

# הנדסת פיצ'רים
df['freq_ratio_31'] = df['ch3_med_freq'] / (df['ch1_med_freq'] + 1e-5)
df['var_diff_31'] = df['ch3_var'] - df['ch1_var']
df['rms_total'] = df['ch1_rms'] + df['ch2_rms'] + df['ch3_rms']

# 3. קידוד (One-Hot)
categorical_cols = ['Hypertension', 'Diabetes', 'Placental_position', 
                    'Bleeding_first_trimester', 'Bleeding_second_trimester', 
                    'Funneling', 'Smoker']
df.replace('None', np.nan, inplace=True)
df = pd.get_dummies(df, columns=[c for c in categorical_cols if c in df.columns], dummy_na=False, drop_first=True)

for col in ['Weight', 'Parity', 'Abortions']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. חלוקה ל-Train ו-Test
df_numeric = df.select_dtypes(include=[np.number])
X_raw = df_numeric.drop(columns=['Label', 'RecID', 'Gestation'], errors='ignore')
y = df_numeric['Label']

imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)

print("Performing RFE...")
rf_selector = RandomForestClassifier(n_estimators=50, random_state=42)
selector = RFE(estimator=rf_selector, n_features_to_select=15)
X_selected = selector.fit_transform(X_imputed, y)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)

def get_ensemble():
    xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, scale_pos_weight=6, subsample=0.8, random_state=42, eval_metric='logloss')
    rf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight='balanced', random_state=42)
    lgbm = LGBMClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, scale_pos_weight=6, subsample=0.8, random_state=42, verbose=-1)
    return VotingClassifier(estimators=[('xgb', xgb), ('rf', rf), ('lgbm', lgbm)], voting='soft')

# 5. Cross-Validation
print("\nStarting 5-Fold Cross Validation on Train set...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_thresholds = []

X_train_np = X_train
y_train_np = y_train.values

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_np, y_train_np)):
    X_fold_train, X_fold_val = X_train_np[train_idx], X_train_np[val_idx]
    y_fold_train, y_fold_val = y_train_np[train_idx], y_train_np[val_idx]
    
    scaler_cv = StandardScaler()
    X_fold_train_scaled = scaler_cv.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler_cv.transform(X_fold_val)
    
    smt_cv = SMOTETomek(random_state=42)
    X_fold_train_res, y_fold_train_res = smt_cv.fit_resample(X_fold_train_scaled, y_fold_train)
    
    ensemble_cv = get_ensemble()
    ensemble_cv.fit(X_fold_train_res, y_fold_train_res)
    
    probs_val = ensemble_cv.predict_proba(X_fold_val_scaled)[:, 1]
    
    precisions, recalls, thresholds = precision_recall_curve(y_fold_val, probs_val)
    f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls + 1e-8)
    best_idx = np.argmax(f2_scores)
    best_idx_thresh = min(best_idx, len(thresholds)-1)
    best_thresh = thresholds[best_idx_thresh]
    
    best_thresholds.append(best_thresh)

optimal_global_threshold = np.mean(best_thresholds)
print(f"Selected Global Threshold: {optimal_global_threshold:.2f}\n")

# 6. אימון המודל הסופי ומבחן
print("Training Final Ensemble Model on the entire Train set...")
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_np)
X_test_scaled = scaler_final.transform(X_test)

smt_final = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt_final.fit_resample(X_train_scaled, y_train_np)

final_model = get_ensemble()
final_model.fit(X_train_res, y_train_res)

probs = final_model.predict_proba(X_test_scaled)[:, 1]
predictions = (probs >= optimal_global_threshold).astype(int)

print(f"\n--- Final Clinical Model Test Results (Threshold: {optimal_global_threshold:.2f}) ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))